# MEN 4338 · Week 3 — Spatial continuity: h-scatterplots and the variogram

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W3_experimental_variogram.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Two holes 25 m apart usually have similar grades; two holes 500 m apart do not. The **variogram** measures
how quickly that similarity is lost with distance. It is the central tool of geostatistics: kriging,
simulation and the classification of resources all start from it.

**What you will practise:** pairs of samples and their distances · h-scatterplots · the semivariance
$\gamma(h)$ · lag classes and tolerances · reading nugget, sill and range · the effect of clustered data.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('tests/w3/q1.py', 'tests/q1.py'), ('tests/w3/q2.py', 'tests/q2.py'), ('tests/w3/q3.py', 'tests/q3.py'), ('tests/w3/q4.py', 'tests/q4.py'), ('tests/w3/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Pairs of samples

With $n$ samples there are $n(n-1)/2$ pairs. NumPy gives all of them at once with `np.triu_indices`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

samples = pd.read_csv("bassam_samples.csv")
grid = samples[samples["campaign"] == "grid"]          # the regular 50 m campaign
x, y, au = grid["x"].to_numpy(), grid["y"].to_numpy(), grid["au_gpt"].to_numpy()

i, j = np.triu_indices(4, 1)
print("pairs of 4 samples:", list(zip(i, j)))

## 2. h-scatterplots

Take every pair separated by roughly a distance $h$, and plot the grade at one end (*tail*) against the
grade at the other (*head*). Close pairs fall near the 1:1 line; distant pairs scatter.

In [ ]:
i, j = np.triu_indices(len(x), 1)
d = np.hypot(x[j] - x[i], y[j] - y[i])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, h in zip(axes, [50, 150, 400]):
    m = np.abs(d - h) < 10
    ax.scatter(au[i][m], au[j][m], s=4, alpha=0.4)
    ax.plot([0, 4], [0, 4], "k--", lw=0.8)
    r = np.corrcoef(au[i][m], au[j][m])[0, 1]
    ax.set_title(f"h ≈ {h} m · {m.sum()} pairs · r = {r:.2f}")
    ax.set_xlabel("tail (g/t)"); ax.set_ylabel("head (g/t)")
plt.tight_layout(); plt.show()

## 3. The semivariogram

The moment of inertia of the h-scatterplot about the 1:1 line is the **semivariance**:

$$\gamma(h) = \frac{1}{2N(h)} \sum_{(i,j)\,:\,d_{ij} \approx h} \left(z_i - z_j\right)^2$$

Data are never exactly $h$ apart, so pairs are grouped into **lag classes**: class $k$ holds the pairs with
$|d - k \cdot lag| < lag/2$. Plot $\gamma$ against the mean distance of each class and read:

* the **nugget** $c_0$: the jump at the origin (assay error, very short-scale variability),
* the **sill**: the plateau, close to the variance of the data,
* the **range** $a$: the distance where the plateau is reached; beyond it, samples are unrelated.

---
## Exercises

Q1 to Q3 use the **grid holes** only (`x`, `y`, `au` defined above).

### Q1. All the pairs

Store the number of pairs in `n_pairs`, the distance of every pair in the array `dist` and the largest
distance in `max_dist`.

In [ ]:
n_pairs = ...
dist = ...
max_dist = ...
n_pairs, max_dist

In [ ]:
grader.check("q1")

### Q2. One lag by hand

Keep the pairs with $40 \le d < 60$ m. Store their number in `n_pairs_50`, the two grades of each pair in
`tail` (first sample) and `head` (second sample), the semivariance in `gamma_50` and the correlation
coefficient of the h-scatterplot in `rho_50`.

In [ ]:
m = ...
n_pairs_50 = ...
tail = ...
head = ...
gamma_50 = ...
rho_50 = ...
n_pairs_50, gamma_50, rho_50

In [ ]:
grader.check("q2")

### Q3. The experimental variogram

Write `exp_variogram(x, y, v, lag, nlags)` returning three arrays of length `nlags`: the mean distance of
the pairs in each class, the semivariance, and the number of pairs (use `np.nan` for an empty class).
Class $k = 1 \dots nlags$ holds the pairs with $|d - k \cdot lag| < lag/2$.

In [ ]:
def exp_variogram(x, y, v, lag, nlags):
    ...

exp_variogram(np.array([0., 10, 20, 30]), np.zeros(4), np.array([1., 2, 4, 7]), 10, 3)

In [ ]:
grader.check("q3")

### Q4. Grid holes versus all holes

Compute the variogram with lag 50 m and 12 lags:

* on the grid holes: `h_grid, g_grid, n_grid`,
* on all samples (grid + infill): `h_all, g_all, n_all`.

Store the variances `var_grid` and `var_all` (pandas `.var()`). Plot both variograms. Why is the variogram
of all samples higher and more erratic? (Think about where the infill holes are, and about the
*proportional effect*: in rich zones grades also vary more.)

In [ ]:
h_grid, g_grid, n_grid = ...
h_all, g_all, n_all = ...
var_grid = ...
var_all = ...

In [ ]:
grader.check("q4")

### Q5. Read the variogram

A lag with fewer than 30 pairs is not reliable: store `reliable = n_grid >= 30`. Plot `g_grid` against
`h_grid` in `fig_vg, ax_vg` with markers, a dashed horizontal line at `var_grid`, axis labels with units.
Store your first reading of the nugget (`nugget_guess = g_grid[0]`) and of the sill (`sill_guess = var_grid`),
and write down the range you see.

In [ ]:
reliable = ...

fig_vg, ax_vg = plt.subplots(figsize=(7, 4))
...
plt.show()

nugget_guess = ...
sill_guess = ...

In [ ]:
grader.check("q5")

### Going further (not checked)

* Draw the **variogram cloud**: every pair as a point (distance, half squared difference). Which pairs are the
  extreme ones?
* Change the lag to 25 m on the grid holes. Why are half of the classes nearly empty?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()